# Import and PIP

In [1]:
# import numpy as np
# import pandas as pd
# # from scipy import stats

# import matplotlib.pyplot as plt
# !pip install pypdf pdfplumber pymupdf

import importlib
import subprocess
import sys

## import chekcer

def _ensure_module(import_name: str, pip_name: str | None = None):
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        if pip_name is None:
            raise
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        return importlib.import_module(import_name)

# stdlib (always available in normal Python environments)
Path = importlib.import_module("pathlib").Path
re = importlib.import_module("re")

# third-party (install only if missing)
pd = _ensure_module("pandas", "pandas")
pdfplumber = _ensure_module("pdfplumber", "pdfplumber")

print("Ready:", Path, pd.__name__, pdfplumber.__name__)

Ready: <class 'pathlib.Path'> pandas pdfplumber


# PDF best file type

In [2]:
from pathlib import Path
from typing import Dict, Any, List, Optional


def _extract_with_pymupdf(pdf_path: Path) -> str:
    import fitz  # pymupdf

    doc = fitz.open(str(pdf_path))
    try:
        parts = [page.get_text("text") or "" for page in doc]
    finally:
        doc.close()
    return "\n".join(parts).strip()


def _extract_with_pypdf(pdf_path: Path) -> str:
    from pypdf import PdfReader

    reader = PdfReader(str(pdf_path))
    parts = [(page.extract_text() or "") for page in reader.pages]
    return "\n".join(parts).strip()


def _extract_with_pdfplumber(pdf_path: Path) -> Dict[str, Any]:
    import pdfplumber

    text_parts: List[str] = []
    tables: List[Dict[str, Any]] = []

    with pdfplumber.open(str(pdf_path)) as pdf:
        for i, page in enumerate(pdf.pages, start=1):
            page_text = page.extract_text() or ""
            page_tables = page.extract_tables() or []
            text_parts.append(page_text)
            tables.append({"page": i, "tables": page_tables})

    return {
        "text": "\n".join(text_parts).strip(),
        "tables": tables
    }


def parse_pdf_smart(pdf_path: str | Path) -> Dict[str, Any]:
    """
    Tries PyMuPDF -> pypdf -> pdfplumber and returns best available text.
    Also returns tables if pdfplumber succeeds.
    """
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    attempts: List[Dict[str, Optional[str]]] = []
    best_text = ""
    best_engine = None
    tables: List[Dict[str, Any]] = []

    # 1) PyMuPDF
    try:
        text = _extract_with_pymupdf(pdf_path)
        attempts.append({"engine": "pymupdf", "error": None})
        if len(text) > len(best_text):
            best_text = text
            best_engine = "pymupdf"
    except Exception as e:
        attempts.append({"engine": "pymupdf", "error": str(e)})

    # 2) pypdf
    try:
        text = _extract_with_pypdf(pdf_path)
        attempts.append({"engine": "pypdf", "error": None})
        if len(text) > len(best_text):
            best_text = text
            best_engine = "pypdf"
    except Exception as e:
        attempts.append({"engine": "pypdf", "error": str(e)})

    # 3) pdfplumber (text + tables)
    try:
        plumber_result = _extract_with_pdfplumber(pdf_path)
        text = plumber_result["text"]
        tables = plumber_result["tables"]
        attempts.append({"engine": "pdfplumber", "error": None})
        if len(text) > len(best_text):
            best_text = text
            best_engine = "pdfplumber"
    except Exception as e:
        attempts.append({"engine": "pdfplumber", "error": str(e)})

    if not best_text:
        errors = [f"{a['engine']}: {a['error']}" for a in attempts if a["error"]]
        raise RuntimeError("All PDF parsers failed:\n" + "\n".join(errors))

    return {
        "engine_used": best_engine,
        "text": best_text,
        "char_count": len(best_text),
        "tables": tables,  # empty if pdfplumber failed or found none
        "attempts": attempts
    }

### Program starts here 


In [ ]:
from pathlib import Path
import re
import pandas as pd
import pdfplumber


def _normalize_text(s: str) -> str:
    """Upper-case and collapse whitespace for robust marker/header matching."""
    return re.sub(r"\s+", " ", (s or "").strip().upper())


def _parse_numeric(series: pd.Series) -> pd.Series:
    """Parse numeric strings like '1,234.56' or '(1,234.56)' into floats."""
    cleaned = (
        series.astype(str)
        .str.replace(",", "", regex=False)
        .str.replace("$", "", regex=False)
        .str.replace("(", "-", regex=False)
        .str.replace(")", "", regex=False)
        .str.strip()
    )
    cleaned = cleaned.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    return pd.to_numeric(cleaned, errors="coerce")


def extract_trading_activity_table(
    pdf_path: str = "2026-03-05.pdf",
    out_csv: str = "securities_trading_activity_table.csv",
    start_marker: str = "SECURITIES TRADING ACTIVITY",
    end_marker: str = "TRADING SUMMARY",
    strict: bool = True,
    max_malformed_rows: int = 0,
    min_completeness: float = 0.70,
):
    """
    Extract the main trading activity table between two section markers with QA checks.

    Built-in validation ensures we captured the right section/table and not partial data:
    - start and end markers both exist in the PDF and are in the right order
    - selected table has required columns
    - malformed rows are controlled
    - output table is not sparse/incomplete
    """
    pdf_path = Path(pdf_path)
    if not pdf_path.exists():
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    start_norm = _normalize_text(start_marker)
    end_norm = _normalize_text(end_marker)

    required_columns = [
        "Symbol & Name",
        "Cusip",
        "Trade Date",
        "Settlement Date",
        "Account Type",
        "Buy/Sell",
        "Quantity",
        "Price",
        "Gross Amount",
        "Commission",
        "Fee/Tax",
        "Net Amount",
        "MKT",
        "Solicitation",
        "CAP",
        "Overnight Trade",
        "Algorithm",
        "Callable",
    ]
    required_columns_norm = [_normalize_text(c) for c in required_columns]

    page_texts = []
    with pdfplumber.open(str(pdf_path)) as doc:
        for page_num, page in enumerate(doc.pages, start=1):
            text = page.extract_text() or ""
            page_texts.append((page_num, text, _normalize_text(text)))

        start_pages = [p for p, _, ntext in page_texts if start_norm in ntext]
        end_pages = [p for p, _, ntext in page_texts if end_norm in ntext]

        if not start_pages:
            raise ValueError(f"Start marker not found: {start_marker}")
        if not end_pages:
            raise ValueError(f"End marker not found: {end_marker}")

        start_page = min(start_pages)
        end_page_candidates = [p for p in end_pages if p >= start_page]
        if not end_page_candidates:
            raise ValueError(
                f"End marker appears before start marker (start page {start_page}, end pages {end_pages})"
            )
        end_page = min(end_page_candidates)

        candidate_tables = []
        for page in doc.pages[start_page - 1:end_page]:
            page_num = page.page_number
            tables = page.extract_tables() or []
            for t_idx, table in enumerate(tables, start=1):
                if not table or len(table) < 2:
                    continue

                header_raw = table[0]
                header_norm = [_normalize_text(str(c)) for c in header_raw]

                matched_required = sum(
                    1 for req in required_columns_norm if any(req == h for h in header_norm)
                )

                row_lengths = [len(r) for r in table if r]
                max_cols = max(row_lengths) if row_lengths else 0

                candidate_tables.append({
                    "page": page_num,
                    "table_index": t_idx,
                    "table": table,
                    "matched_required": matched_required,
                    "max_cols": max_cols,
                })

    if not candidate_tables:
        raise RuntimeError(
            f"No tables found between markers on pages {start_page} to {end_page}."
        )

    candidate_tables.sort(
        key=lambda x: (x["matched_required"], x["max_cols"], len(x["table"])),
        reverse=True,
    )
    chosen = candidate_tables[0]
    chosen_table = chosen["table"]
    chosen_page = chosen["page"]

    if chosen["matched_required"] < 10:
        raise RuntimeError(
            "Could not confidently identify the trading activity table: "
            f"only matched {chosen['matched_required']} required columns."
        )

    header = [str(c).strip() for c in chosen_table[0]]
    n_cols = len(header)

    # Keep all matching tables in page order so multi-page sections are merged.
    ordered_candidates = sorted(candidate_tables, key=lambda x: (x["page"], x["table_index"]))

    selected_tables = []
    for candidate in ordered_candidates:
        include = False
        if int(candidate["matched_required"]) >= 10:
            include = True
        else:
            # Some page breaks produce continuation tables without a repeated header.
            if int(candidate["page"]) > chosen_page and max(1, n_cols - 1) <= int(candidate["max_cols"]) <= n_cols + 1:
                include = True
        if include:
            selected_tables.append(candidate)

    if not selected_tables:
        selected_tables = [chosen]

    malformed_rows = 0
    cleaned_rows = []
    selected_pages = sorted({int(t["page"]) for t in selected_tables})
    best_header_match = max(int(t["matched_required"]) for t in selected_tables)
    for candidate in selected_tables:
        table = candidate["table"]
        start_index = 1 if int(candidate["matched_required"]) >= 10 else 0
        for row in table[start_index:]:
            row = row or []
            if len(row) != n_cols:
                malformed_rows += 1
            if len(row) < n_cols:
                row = row + [None] * (n_cols - len(row))
            elif len(row) > n_cols:
                row = row[:n_cols]
            cleaned_rows.append(row)

    if strict and malformed_rows > max_malformed_rows:
        raise RuntimeError(
            f"Malformed rows detected: {malformed_rows} (allowed: {max_malformed_rows})."
        )

    df = pd.DataFrame(cleaned_rows, columns=header)

    for col in df.columns:
        df[col] = df[col].map(
            lambda x: x.replace("\n", " ").strip() if isinstance(x, str) else x
        )
    df.replace("", pd.NA, inplace=True)
    df.dropna(axis=0, how="all", inplace=True)
    df.dropna(axis=1, how="all", inplace=True)
    df.reset_index(drop=True, inplace=True)

    if df.empty:
        raise RuntimeError("Extracted table is empty after cleanup.")

    # Enforce canonical schema so output always has all required columns, including Callable.
    normalized_df_cols = [_normalize_text(c) for c in df.columns]
    norm_to_original = {n: c for n, c in zip(normalized_df_cols, df.columns)}

    for req_display, req_norm in zip(required_columns, required_columns_norm):
        if req_norm in norm_to_original:
            src_col = norm_to_original[req_norm]
            if src_col != req_display:
                df.rename(columns={src_col: req_display}, inplace=True)
        else:
            df[req_display] = pd.NA

    ordered_cols = required_columns + [c for c in df.columns if c not in required_columns]
    df = df[ordered_cols]

    normalized_df_cols = [_normalize_text(c) for c in df.columns]
    missing_required = [
        req for req in required_columns_norm if req not in normalized_df_cols
    ]
    if strict and missing_required:
        raise RuntimeError(
            "Missing expected columns after extraction: " + ", ".join(missing_required)
        )

    # Compute overall table completeness as:
    # (number of non-null cells) / (total number of cells)
    total_cells = df.shape[0] * df.shape[1]
    non_null_cells = int(df.notna().sum().sum())
    completeness = non_null_cells / total_cells if total_cells else 0.0

    # In strict mode, fail fast if the extracted table is too sparse.
    # This protects against partial/incorrect table captures.
    if strict and completeness < min_completeness:
        raise RuntimeError(
            f"Table completeness too low: {completeness:.2%} < {min_completeness:.2%}"
        )

    if strict:
        for req in required_columns:
            if df[req].isna().all() and req not in ("Callable", "Cusip"):
                raise RuntimeError(f"Required column has no values: {req}")

    numeric_totals = {}
    total_targets = {
        "quantity_total": "Quantity",
        "gross_amount_total": "Gross Amount",
        "net_amount_total": "Net Amount",
    }
    for total_key, target_col in total_targets.items():
        if target_col in df.columns:
            numeric_totals[total_key] = float(_parse_numeric(df[target_col]).sum(skipna=True))

    df.to_csv(out_csv, index=False)

    qa_report = {
        "saved_file": out_csv,
        "source_page": chosen_page,
        "source_pages": selected_pages,
        "shape": (int(df.shape[0]), int(df.shape[1])),
        "malformed_rows": malformed_rows,
        "completeness": round(completeness, 4),
        "matched_required_columns": best_header_match,
        "numeric_totals": numeric_totals,
    }

    print(f"Saved: {qa_report['saved_file']}")
    print(f"Source page: {qa_report['source_page']}")
    print(f"Source pages: {qa_report['source_pages']}")
    print(f"Shape: {qa_report['shape'][0]} rows x {qa_report['shape'][1]} cols")
    print(f"Malformed rows: {qa_report['malformed_rows']}")
    print(f"Completeness: {qa_report['completeness']:.2%}")
    print(f"Matched required columns: {qa_report['matched_required_columns']}")
    if numeric_totals:
        print(f"Numeric totals: {numeric_totals}")
    print(df[required_columns].head(5).to_string(index=False))

    return df, qa_report


# df, qa = extract_trading_activity_table()
# qa

In [4]:
# here1
# Optional interactive runner: enter date or PDF filename at runtime.
pdf_file = input("Enter PDF filename or date (default: 2026-03-05): ").strip() or "2026-03-05"
if not pdf_file.lower().endswith(".pdf"):
    pdf_file += ".pdf"

pdf_path = Path(pdf_file)
if not pdf_path.exists():
    raise FileNotFoundError(f"PDF not found: {pdf_path}")

out_csv = f"{pdf_path.stem}_securities_trading_activity_table.csv"
df, qa = extract_trading_activity_table(
    pdf_path=str(pdf_path),
    out_csv=out_csv,
    strict=True,
    max_malformed_rows=0,
    min_completeness=0.70,
)


print(f"Created: {out_csv}")


Saved: 2026-04-09_securities_trading_activity_table.csv
Source page: 3
Shape: 15 rows x 18 cols
Malformed rows: 0
Completeness: 92.96%
Matched required columns: 18
Numeric totals: {'quantity_total': 50.0, 'gross_amount_total': -8221.65, 'net_amount_total': -8223.540000000003}
           Symbol & Name     Cusip Trade Date Settlement Date Account Type Buy/Sell Quantity  Price Gross Amount Commission Fee/Tax Net Amount MKT Solicitation CAP Overnight Trade Algorithm Callable
COIN COINBASE GLOBAL INC 19260Q107 04/09/2026      04/10/2026            M        B    25.00 174.97    -4,374.25       0.00    0.00  -4,374.25 OTH            U   A               N         N     <NA>
COIN COINBASE GLOBAL INC 19260Q107 04/09/2026      04/10/2026            M        S   -75.00 175.67    13,175.10       0.00   -0.29  13,174.81 OTH            U   A               N         N     <NA>
COIN COINBASE GLOBAL INC 19260Q107 04/09/2026      04/10/2026            M        B    50.00 175.83    -8,791.50       0.00   